In [2]:
# %% Complete Fixed Driver Drowsiness Detection System
import cv2
import numpy as np
import time
import threading
import sys
import platform

# %% Import MediaPipe properly
import mediapipe as mp

mp_face_mesh = mp.solutions.face_mesh
mp_drawing = mp.solutions.drawing_utils
drawing_spec = mp_drawing.DrawingSpec(thickness=1, circle_radius=1, color=(0, 255, 0))

# %% Sound alert setup
def play_alert_sound():
    """Play alert sound based on operating system"""
    system = platform.system()
    try:
        if system == "Windows":
            import winsound
            winsound.Beep(1000, 500)
        elif system == "Darwin":  # macOS
            import os
            os.system('afplay /System/Library/Sounds/Ping.aiff')
        else:  # Linux
            import os
            os.system('paplay /usr/share/sounds/freedesktop/stereo/complete.oga 2>/dev/null || echo -e "\a"')
    except:
        pass

# %% Initialize Face Mesh
face_mesh = mp_face_mesh.FaceMesh(
    static_image_mode=False,
    max_num_faces=1,
    refine_landmarks=True,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)

# %% Eye landmark indices
LEFT_EYE = [33, 133, 159, 145]   # Left: corner_left, corner_right, top, bottom
RIGHT_EYE = [362, 263, 386, 374] # Right: corner_left, corner_right, top, bottom

# %% Eye Aspect Ratio function
def eye_aspect_ratio(eye_landmarks):
    """Calculate the Eye Aspect Ratio (EAR)"""
    left_corner = np.array(eye_landmarks[0])
    right_corner = np.array(eye_landmarks[1])
    top = np.array(eye_landmarks[2])
    bottom = np.array(eye_landmarks[3])
    
    horizontal_dist = np.linalg.norm(right_corner - left_corner)
    vertical_dist = np.linalg.norm(bottom - top)
    
    if horizontal_dist > 0:
        return vertical_dist / horizontal_dist
    return 0

# %% Detection parameters - ADJUSTED FOR BETTER RESET
EYE_CLOSED_THRESH = 0.22      # EAR threshold for closed eyes
DROWSY_FRAMES = 20            # Frames needed to trigger alert
RESET_FRAMES = 15             # Frames with eyes open to reset alert (FIXED!)

# Counters
drowsy_counter = 0            # Counts consecutive frames with eyes closed
open_eye_counter = 0          # Counts consecutive frames with eyes open (NEW!)
alert_active = False
total_blinks = 0
drowsy_events = 0             # Track total number of drowsiness events

# %% Camera initialization
print("=" * 60)
print("Initializing Driver Drowsiness Detection System...")
print("=" * 60)

cap = None

# Try to open camera
for camera_index in [0, 1]:
    if platform.system() == "Windows":
        cap = cv2.VideoCapture(camera_index, cv2.CAP_DSHOW)
    else:
        cap = cv2.VideoCapture(camera_index)
    
    if cap.isOpened():
        ret, test_frame = cap.read()
        if ret and test_frame is not None:
            print(f"✓ Camera {camera_index} opened successfully")
            print(f"  Resolution: {int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))}x{int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))}")
            break
        else:
            cap.release()
            cap = None

if not cap or not cap.isOpened():
    print("\n❌ ERROR: Could not access any camera!")
    sys.exit(1)

cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)

print("\n" + "=" * 60)
print("SYSTEM READY!")
print("Press 'q' to quit | Press 'r' to reset counter")
print("=" * 60 + "\n")

# %% Main detection loop
frame_count = 0
fps_start_time = time.time()
fps = 0

try:
    while True:
        # Read frame
        success, frame = cap.read()
        if not success:
            print("Warning: Could not read frame")
            continue
        
        frame_count += 1
        
        # Flip horizontally for mirror view
        frame = cv2.flip(frame, 1)
        h, w = frame.shape[:2]
        
        # Convert BGR to RGB
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        
        # Process with MediaPipe
        results = face_mesh.process(rgb_frame)
        
        ear = None
        face_detected = False
        
        if results.multi_face_landmarks:
            face_detected = True
            face_landmarks = results.multi_face_landmarks[0]
            
            # Extract eye coordinates
            left_eye_pts = []
            for idx in LEFT_EYE:
                landmark = face_landmarks.landmark[idx]
                x = int(landmark.x * w)
                y = int(landmark.y * h)
                left_eye_pts.append((x, y))
                cv2.circle(frame, (x, y), 3, (0, 255, 255), -1)
            
            right_eye_pts = []
            for idx in RIGHT_EYE:
                landmark = face_landmarks.landmark[idx]
                x = int(landmark.x * w)
                y = int(landmark.y * h)
                right_eye_pts.append((x, y))
                cv2.circle(frame, (x, y), 3, (0, 255, 255), -1)
            
            # Calculate EAR
            left_ear = eye_aspect_ratio(left_eye_pts)
            right_ear = eye_aspect_ratio(right_eye_pts)
            ear = (left_ear + right_ear) / 2.0
            
            # Draw eye contours
            for eye_pts in [left_eye_pts, right_eye_pts]:
                cv2.line(frame, eye_pts[0], eye_pts[1], (0, 255, 0), 1)
                cv2.line(frame, eye_pts[2], eye_pts[3], (0, 255, 0), 1)
            
            # ===== FIXED LOGIC: Proper alert reset =====
            if ear < EYE_CLOSED_THRESH:
                # Eyes are closed
                drowsy_counter += 1
                open_eye_counter = 0  # Reset open eye counter
                
                # Check if drowsiness threshold reached
                if drowsy_counter >= DROWSY_FRAMES:
                    if not alert_active:
                        alert_active = True
                        drowsy_events += 1
                        print(f"⚠️  DROWSINESS ALERT #{drowsy_events}! EAR: {ear:.3f} | Time: {time.strftime('%H:%M:%S')}")
                        threading.Thread(target=play_alert_sound, daemon=True).start()
            else:
                # Eyes are open
                if drowsy_counter > 2:  # Was a valid blink
                    total_blinks += 1
                
                # Reset drowsy counter since eyes are open
                drowsy_counter = 0
                
                # Count consecutive frames with eyes open
                open_eye_counter += 1
                
                # Reset alert after enough frames with eyes open
                if alert_active and open_eye_counter >= RESET_FRAMES:
                    alert_active = False
                    print(f"✓ Alert reset - Eyes open for {RESET_FRAMES} frames | Time: {time.strftime('%H:%M:%S')}")
                    open_eye_counter = 0  # Reset counter after clearing alert
        else:
            # No face detected
            drowsy_counter = max(0, drowsy_counter - 1)
            open_eye_counter = 0  # Reset since no face
            
            # Don't reset alert immediately if face is lost briefly
            if drowsy_counter == 0:
                # Face lost for too long
                if alert_active and open_eye_counter == 0:
                    # Keep alert active for a moment but don't reset
                    pass
        
        # Calculate FPS
        if frame_count % 10 == 0:
            fps = frame_count / (time.time() - fps_start_time)
        
        # ===== Display Information =====
        # FPS
        cv2.putText(frame, f"FPS: {fps:.1f}", (10, 30),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
        
        # EAR value
        if ear is not None:
            ear_color = (0, 255, 0) if ear >= EYE_CLOSED_THRESH else (0, 0, 255)
            cv2.putText(frame, f"EAR: {ear:.2f}", (10, 60),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.7, ear_color, 2)
            
            # Eye status
            eye_status = "OPEN" if ear >= EYE_CLOSED_THRESH else "CLOSED"
            cv2.putText(frame, f"Eyes: {eye_status}", (10, 90),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.7, ear_color, 2)
        
        # Blink counter
        cv2.putText(frame, f"Blinks: {total_blinks}", (10, 120),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 0), 2)
        
        # Drowsiness events counter
        cv2.putText(frame, f"Alerts: {drowsy_events}", (10, 150),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)
        
        # Drowsiness progress bar
        bar_width = 200
        bar_height = 20
        bar_x = w - bar_width - 20
        bar_y = 30
        
        progress = min(drowsy_counter / DROWSY_FRAMES, 1.0)
        
        cv2.rectangle(frame, (bar_x, bar_y), (bar_x + bar_width, bar_y + bar_height),
                     (255, 255, 255), 2)
        
        if progress > 0:
            bar_color = (0, 0, 255) if progress > 0.5 else (0, 255, 255)
            cv2.rectangle(frame, (bar_x + 2, bar_y + 2),
                         (bar_x + int(bar_width * progress) - 2, bar_y + bar_height - 2),
                         bar_color, -1)
        
        cv2.putText(frame, f"Drowsiness: {int(progress*100)}%",
                   (bar_x, bar_y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5,
                   (255, 255, 255), 1)
        
        # Alert reset progress bar (shows how close to resetting)
        if alert_active:
            reset_progress = min(open_eye_counter / RESET_FRAMES, 1.0)
            reset_bar_y = bar_y + bar_height + 10
            
            cv2.rectangle(frame, (bar_x, reset_bar_y), 
                         (bar_x + bar_width, reset_bar_y + bar_height),
                         (255, 255, 255), 2)
            
            if reset_progress > 0:
                cv2.rectangle(frame, (bar_x + 2, reset_bar_y + 2),
                             (bar_x + int(bar_width * reset_progress) - 2, 
                              reset_bar_y + bar_height - 2),
                             (0, 255, 0), -1)
            
            cv2.putText(frame, f"Resetting: {int(reset_progress*100)}%",
                       (bar_x, reset_bar_y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5,
                       (0, 255, 0), 1)
        
        # Face detection status
        face_status = "FACE DETECTED" if face_detected else "NO FACE"
        face_color = (0, 255, 0) if face_detected else (0, 0, 255)
        cv2.putText(frame, face_status, (10, h - 20),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.7, face_color, 2)
        
        # Drowsiness Alert Display
        if alert_active:
            # Red border overlay
            overlay = frame.copy()
            cv2.rectangle(overlay, (0, 0), (w, h), (0, 0, 255), 15)
            cv2.addWeighted(overlay, 0.4, frame, 0.6, 0, frame)
            
            # Blinking warning text
            if int(time.time() * 2) % 2 == 0:
                alert_text = "⚠️  DROWSINESS DETECTED!  ⚠️"
                text_size = cv2.getTextSize(alert_text, cv2.FONT_HERSHEY_SIMPLEX, 1.2, 3)[0]
                text_x = (w - text_size[0]) // 2
                text_y = h // 2
                
                cv2.rectangle(frame, 
                            (text_x - 10, text_y - text_size[1] - 10),
                            (text_x + text_size[0] + 10, text_y + 10),
                            (0, 0, 0), -1)
                
                cv2.putText(frame, alert_text, (text_x, text_y),
                           cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 0, 255), 3)
            
            # Sub-text with reset counter
            sub_text = f"KEEP EYES OPEN TO RESET ({open_eye_counter}/{RESET_FRAMES})"
            sub_size = cv2.getTextSize(sub_text, cv2.FONT_HERSHEY_SIMPLEX, 0.7, 2)[0]
            sub_x = (w - sub_size[0]) // 2
            cv2.putText(frame, sub_text, (sub_x, h // 2 + 40),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
        
        # Display the frame
        cv2.imshow('Driver Drowsiness Detection System', frame)
        
        # Handle key presses
        key = cv2.waitKey(1) & 0xFF
        if key == ord('q'):
            print("\nQuitting...")
            break
        elif key == ord('r'):
            print("Manual reset...")
            drowsy_counter = 0
            open_eye_counter = 0
            alert_active = False
        elif key == ord('s'):
            timestamp = time.strftime("%Y%m%d_%H%M%S")
            filename = f"drowsiness_screenshot_{timestamp}.jpg"
            cv2.imwrite(filename, frame)
            print(f"Screenshot saved: {filename}")

except KeyboardInterrupt:
    print("\n\nInterrupted by user")
except Exception as e:
    print(f"\n❌ Error: {e}")
    import traceback
    traceback.print_exc()
finally:
    print("\nCleaning up...")
    cap.release()
    cv2.destroyAllWindows()
    face_mesh.close()
    print(f"Total blinks detected: {total_blinks}")
    print(f"Total drowsiness events: {drowsy_events}")
    print(f"Program ended successfully")

Initializing Driver Drowsiness Detection System...
✓ Camera 0 opened successfully
  Resolution: 640x480

SYSTEM READY!
Press 'q' to quit | Press 'r' to reset counter

⚠️  DROWSINESS ALERT #1! EAR: 0.195 | Time: 19:17:33
✓ Alert reset - Eyes open for 15 frames | Time: 19:17:36
⚠️  DROWSINESS ALERT #2! EAR: 0.195 | Time: 19:17:41
✓ Alert reset - Eyes open for 15 frames | Time: 19:17:43
⚠️  DROWSINESS ALERT #3! EAR: 0.124 | Time: 19:17:53
✓ Alert reset - Eyes open for 15 frames | Time: 19:17:56
⚠️  DROWSINESS ALERT #4! EAR: 0.126 | Time: 19:18:01
✓ Alert reset - Eyes open for 15 frames | Time: 19:18:04
⚠️  DROWSINESS ALERT #5! EAR: 0.133 | Time: 19:18:12
✓ Alert reset - Eyes open for 15 frames | Time: 19:18:15
⚠️  DROWSINESS ALERT #6! EAR: 0.185 | Time: 19:18:38
✓ Alert reset - Eyes open for 15 frames | Time: 19:18:41
⚠️  DROWSINESS ALERT #7! EAR: 0.176 | Time: 19:19:01
✓ Alert reset - Eyes open for 15 frames | Time: 19:19:08

Quitting...

Cleaning up...
Total blinks detected: 66
Total dr

In [1]:
import numpy as np
import cv2
import mediapipe as mp
print("NumPy:", np.__version__)
print("OpenCV:", cv2.__version__)
print("MediaPipe:", mp.__version__)


NumPy: 1.26.4
OpenCV: 4.11.0
MediaPipe: 0.10.9
